# Chapter 7 - Lab 2: Leader-follower routing with the OpenAI Agents SDK (customer-care edition)

**Goal**: a **manager agent** routes each request from an enterprise customer's account team to the right
**specialist**, exactly like a contact-centre IVR or triage desk.

*Finance edition*: a manager routed to Stock Price, Company Info and Income Statement agents.
*Customer-care edition*: the manager routes to **SLA Performance**, **Account Profile** and **Billing** specialists.

Two OpenAI Agents SDK mechanisms are compared:
* **Handoffs** (leader-follower): the manager *transfers* the conversation to one specialist.
* **Agents-as-tools** (orchestrator-worker): the manager *calls* several specialists and synthesises.

**Tech stack**: OpenAI Agents SDK (`openai-agents`), OpenAI `gpt-4o`, stub data in `common.py`.

## 1. Install packages

In [ ]:
%pip install -q openai-agents pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%207/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import stub_lookup
print("Stub data loaded.")

## 4. Define the three specialist tools

In [ ]:
def get_sla_report(account_id: str) -> dict:
    """Quarter SLA report: tickets, open P1 incidents, response/resolution SLA met, breaches, top issue."""
    return stub_lookup(account_id)["sla_report"]


def get_account_profile(account_id: str) -> dict:
    """Account profile: company, industry, sites, segment, annual contract value, renewal date, CSM."""
    return stub_lookup(account_id)["profile"]


def get_billing_summary(account_id: str) -> dict:
    """Billing: open invoices, overdue amount and days, service credits issued, last payment, disputes."""
    return stub_lookup(account_id)["billing"]

## 5. Define the three specialist worker agents

In [ ]:
from agents import Agent, Runner, function_tool

MODEL = "gpt-4o"

sla_agent = Agent(
    name="SLA Performance Agent",
    instructions="Report an account's SLA performance: tickets, open P1s, SLA met, breaches and the top issue.",
    tools=[function_tool(get_sla_report)],
    model=MODEL,
)
profile_agent = Agent(
    name="Account Profile Agent",
    instructions="Report an account's profile: company, industry, sites, contract value, renewal date, CSM.",
    tools=[function_tool(get_account_profile)],
    model=MODEL,
)
billing_agent = Agent(
    name="Billing Agent",
    instructions="Report an account's billing status: open and overdue invoices, credits issued, disputes.",
    tools=[function_tool(get_billing_summary)],
    model=MODEL,
)

## 6. The manager agent (leader) with handoffs

The manager has no tools: its job is **routing**. `handoffs` lists the workers it may transfer to; the LLM picks one at runtime.

In [ ]:
manager_agent = Agent(
    name="Manager Agent",
    instructions=("You are the triage desk for NovaMart's enterprise accounts. Determine which specialist is "
                  "best suited to handle the request and transfer the conversation to that agent."),
    handoffs=[sla_agent, profile_agent, billing_agent],
    model=MODEL,
)

## 7. Run and inspect the routing decision

In [ ]:
for query in ["What is the billing status of AC-HOTELS? Is anything overdue?",
              "How did AC-RETAIL perform against its SLA last quarter?",
              "When does AC-CLINICS renew and who is their customer-success manager?"]:
    result = await Runner.run(manager_agent, query)
    print(f"Q: {query}\nRouted to: {result.last_agent.name}\nA: {result.final_output}\n" + "-" * 60)

## 8. Variant: agents-as-tools for a synthesis question

A question that needs **all three** specialists ("Should we worry about AC-HOTELS?") is better served by a manager that *calls* the specialists as tools and keeps control to synthesise.

In [ ]:
account_manager = Agent(
    name="Account Health Manager",
    instructions=("Produce a short account-health briefing for the requested enterprise account. Call every "
                  "specialist tool you need, then synthesise: overall status (HEALTHY | WATCH | AT RISK | "
                  "CRITICAL), the three facts that matter most, and the next action for the account team."),
    tools=[
        sla_agent.as_tool(tool_name="sla_performance", tool_description="SLA performance of an account."),
        profile_agent.as_tool(tool_name="account_profile", tool_description="Profile and renewal of an account."),
        billing_agent.as_tool(tool_name="billing_status", tool_description="Billing status of an account."),
    ],
    model=MODEL,
)
result = await Runner.run(account_manager, "Should we worry about AC-HOTELS ahead of its renewal?")
print(result.final_output)

## 9. Results

**What to notice**
* **Dynamic routing**: adding a specialist (e.g. a Field Service agent) is one new `Agent` and one list entry.
* **Handoff vs as_tool**: a handoff *transfers* the conversation (good for "answer this one thing");
  `as_tool` keeps the manager in control (good for "combine several things").
* **Trade-off**: routing quality depends on the manager's prompt; vague instructions send customers to the wrong desk.
  Chapter 12 adds guardrails for production.